# Reproducibility copy of original Colab notebook
This notebook preserves the authors' analytical workflow, with **saved outputs removed** and credentials/path references sanitized.
It has not been validated end-to-end in a clean environment. Some cells depend on legacy intermediate variables, remote IUCN access, or private screening exports. See `docs/reproducibility.md`.


In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / 'data').is_dir():
    ROOT = ROOT.parent
url = ROOT / 'data' / 'processed' / 'quality_assessment.csv'


In [ ]:
choice_to_score = {
    "0: Not reported/insufficient": 0,
    "1: Partially justified": 1,
    "2: Fully justified with rationale": 2,

    "0: Not addressed": 0,
    "1: Acknowledged but not handled": 1,
    "2: Properly addressed with methods": 2,

    "0: Not described": 0,
    "1: Partially described": 1,
    "2: Fully described": 2,

    "0: No justification": 0,
    "1: Partial justification": 1,
    "2: Comprehensive justification": 2,

    "1: Basic justification": 1,
    "2: Full comparison": 2,

    "0: No validation": 0,
    "1: Basic validation": 1,
    "2: Rigorous validation": 2,

    "0: Incomplete": 0,
    "1: Basic description": 1,
    "2: Detailed description": 2,

    "0: Not available": 0,
    "1: Partially available": 1,
    "2: Fully available with documentation": 2,

    "1: Available upon request": 1,
    "2: Publicly available with documentation": 2,

    "0: Not documented": 0,
    "1: Partially documented": 1,
    "2: Fully documented": 2,

    "0: No comparison": 0,
    "1: Limited comparison": 1,
    "2: Comprehensive comparison": 2,

    "0: Basic metrics only": 0,
    "1: Multiple metrics": 1,
    "2: Full evaluation with analysis": 2,

    "0: Not discussed": 0,
    "1: Briefly mentioned": 1,
    "2: Fully analyzed": 2,

    "1: Briefly mentioned": 1,
    "2: Detailed discussion": 2
}

In [ ]:
words_to_remove = [
    "Paper information",
    "Data quality",
    "Method practices",
    "Transparency",
    "Research completeness",
    "Overall evaluation"
]

In [ ]:
qual_assessment_data = pd.read_csv(url)

In [ ]:
qual_assessment_data = qual_assessment_data.loc[:, ~qual_assessment_data.columns.str.contains("Rationale", case=False)]

In [ ]:
def clean_column(col_name):
    for word in words_to_remove:
        if col_name.startswith(word) and col_name != "Data quality control":
            col_name = col_name.replace(word, "").strip()
    return col_name

In [ ]:
papers_scores = qual_assessment_data.replace(choice_to_score)

In [ ]:
papers_scores.columns = [clean_column(col) for col in papers_scores.columns]

In [ ]:
idx = papers_scores.columns.get_loc("Future directions") + 1
papers_scores = papers_scores.iloc[:, :idx]

In [ ]:
non_score_cols = ['id', 'Title', 'Authors', 'Year', 'DOI/Link']
score_columns = ['Data volume', 'Data balance', 'Data quality control', 'Model selection', 'Model validation', 'Method description', 'Code availability', 'Data availability', 'Technical specifications', 'Comparative analysis', 'Performance evaluation', 'Ecological relevance', 'Future directions']

In [ ]:
papers_scores['Total Score'] = papers_scores[score_columns].sum(axis=1)

In [ ]:
papers_scores.head()

In [ ]:
plt.figure(figsize=(8,6))
sns.histplot(papers_scores['Total Score'], kde=True, color='skyblue', bins=15)
plt.title('Distribution of Total Scores')
plt.xlabel('Total Score')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


In [ ]:
category_columns = ['Data volume', 'Data balance', 'Data quality control',
                    'Model selection', 'Model validation',
                    'Method description', 'Code availability', 'Data availability', 'Technical specifications',
                    'Comparative analysis', 'Performance evaluation', 'Ecological relevance', 'Future directions']


category_scores = papers_scores[category_columns].mean(axis=0)

In [ ]:
# Normalize score data for heatmap plotting
heatmap_data = papers_scores[category_columns]
plt.figure(figsize=(10,8))
sns.heatmap(heatmap_data, annot=True, cmap='coolwarm', cbar=False)
plt.title('Scores Heatmap by Category for Each Paper')
plt.xlabel('Category')
plt.ylabel('Paper')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
category_scores.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Average Scores by Category')
plt.ylabel('Average Score')
plt.xlabel('Category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,6))
sns.violinplot(data=papers_scores[category_columns], palette="muted")
plt.title('Distribution of Scores by Category')
plt.xlabel('Category')
plt.ylabel('Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:


# Select a paper for demonstration, for example Paper_1
paper_scores = papers_scores.iloc[0][category_columns].values

# Categories (angles)
categories = category_columns
num_categories = len(categories)

# Compute the angle for each axis of the radar chart
angles = np.linspace(0, 2 * np.pi, num_categories, endpoint=False).tolist()

# Make the plot a circle
paper_scores = np.concatenate((paper_scores, [paper_scores[0]]))
angles += angles[:1]

# Plot
plt.figure(figsize=(8,8))
ax = plt.subplot(111, polar=True)
ax.fill(angles, paper_scores, color='cyan', alpha=0.25)
ax.plot(angles, paper_scores, color='blue', linewidth=2)

# Add labels and title
ax.set_yticklabels([])
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, rotation=45, ha='right');
# plt.title(f"Radar Chart for Paper: {papers_scores['Title'].iloc[0]}")
# plt.tight_layout()

In [ ]:
selected_columns_for_transparency_gaps = papers_scores[[
    'id',
    'Method description',
    'Code availability',
    'Data availability',
    'Technical specifications',
]]

# Save to a comma-separated text file
selected_columns_for_transparency_gaps.to_csv('transparency_gaps_analysis_data.txt', index=False, sep=',')